In [ ]:
!wget https://huggingface.co/datasets/barryallen16/java_vulnerability_dataset/resolve/main/finetuning_dataset_v3.jsonl

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "Qwen/Qwen2.5-Coder-7B-Instruct",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing=True, 
    random_state=1611,
    use_rslora=False,
    loftq_config=None,
)

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template =  'qwen-2.5',
)

In [ ]:
from datasets import load_dataset
dataset = load_dataset("json", data_files="finetuning_dataset_enhanced.jsonl", split="train")

In [ ]:
def formatting_prompts_func(examples):
    texts = []
    messages_list = examples["messages"]
    if isinstance(messages_list[0], dict):
        text = tokenizer.apply_chat_template(
            messages_list,
            tokenize=False,
            add_generation_prompt=False
        )
        texts.append(text)
    else:
        for messages in messages_list:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=False
            )
            texts.append(text)
    return texts

In [ ]:
test_example = dataset[0]
formatted = formatting_prompts_func(test_example)
print(formatted)

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    formatting_func=formatting_prompts_func,
    tokenizer = tokenizer,
    train_dataset=dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 1611,
        output_dir = "outputs",
        report_to = "none",
        save_strategy = "steps",
        save_steps = 10,
          max_steps = 60,
    ),

)

In [ ]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

In [ ]:
from unsloth import unsloth_train
trainer_stats = unsloth_train(trainer)

In [ ]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

In [ ]:
# @title
FastLanguageModel.for_inference(model)
code = """```java
import java.util.Random;

public class NonVulnerableClass {
    public int getRandomNumber() {
        Random random = new Random(); // Uses system time (non-fixed seed)
        return random.nextInt();
    }
}
```"""

##expected:
##"Classification: NOT VULNERABLE. Vulnerability Type: "  # (Blank if no vulnerability)
messages=[
                {
                    "role": "system",
                    "content": "You are a code vulnerability classifier. Analyze the provided Java code and classify it as either 'VULNERABLE' or 'NOT VULNERABLE'. If vulnerable, specify the exact vulnerability type from the dataset. Respond strictly in this format: 'Classification: <VULNERABLE/NOT VULNERABLE>. Vulnerability Type: <type>'. Do not include reasoning.",
                },
                {"role": "user", "content": code},
            ]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict=True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],streamer = text_streamer,
                   max_new_tokens = 256,
    )

In [ ]:
# @title
FastLanguageModel.for_inference(model)
code = """```java
import java.util.Random;

public class VulnerableClass {
    private static final long SEED = 1234567890L;

    public int getRandomNumber() {
        Random random = new Random(SEED);
        return random.nextInt();
    }
}
```"""
##expected:
##"Classification: VULNERABLE. Vulnerability Type: Improper Control of Generation of Code (\"Code Injection\")"
messages=[
                {
                    "role": "system",
                    "content": "You are a code vulnerability classifier. Analyze the provided Java code and classify it as either 'VULNERABLE' or 'NOT VULNERABLE'. If vulnerable, specify the exact vulnerability type from the dataset. Respond strictly in this format: 'Classification: <VULNERABLE/NOT VULNERABLE>. Vulnerability Type: <type>'. Do not include reasoning.",
                },
                {"role": "user", "content": code},
            ]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize = True,
    add_generation_prompt = True,
    return_tensors = "pt",
    return_dict=True,
).to("cuda")

from transformers import TextStreamer
text_streamer = TextStreamer(tokenizer, skip_prompt = True)
_ = model.generate(input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],streamer = text_streamer,
                   max_new_tokens = 256,
    )

In [ ]:
# @title
testcases = [
    {
        "code": 'public class Example8 {\n    public static void main(String[] args) {\n        String password = "secret";\n        System.out.println(password);\n    }\n}',
        "expected": "VULNERABLE",
        "description": "Hardcoded credentials vulnerability",
    },
    {
        "code": 'public class Example17 {\n    public static void main(String[] args) throws Exception {\n        String fileName = args[0];\n        if (!fileName.matches("[a-zA-Z0-9]+\\\\.txt")) {\n            throw new IllegalArgumentException("Invalid file name");\n        }\n        File file = new File("uploads/" + fileName);\n        FileInputStream fis = new FileInputStream(file);\n    }\n}',
        "expected": "NOT_VULNERABLE",
        "description": "Validated file name prevents path traversal",
    },
    {
        "code": 'public class Example23 {\n    public void processRequest(HttpServletRequest request) throws Exception {\n        String token = request.getParameter("token");\n        String storedToken = System.getenv("AUTH_TOKEN");\n        if (token.equals(storedToken)) {\n            // Grant access\n        }\n    }\n}',
        "expected": "NOT_VULNERABLE",
        "description": "Secure token comparison using environment variable",
    },
    {
        "code": 'public class Example30 {\n    public String hashPassword(String password) throws Exception {\n        MessageDigest md = MessageDigest.getInstance("SHA-1");\n        return new String(md.digest(password.getBytes()));\n    }\n}',
        "expected": "VULNERABLE",
        "description": "Using weak hashing algorithm (SHA-1) for passwords",
    },
    {
        "code": "public class Example61 {\n    public void setHeaders(HttpServletResponse response) {\n        // No security headers set\n    }\n}",
        "expected": "VULNERABLE",
        "description": "Missing security headers allowing XSS and other attacks",
    },
    {
        "code": "public class Example86 {\n    public boolean compareSecrets(String a, String b) {\n        return a.equals(b);\n    }\n}",
        "expected": "VULNERABLE",
        "description": "Timing attack vulnerability in string comparison",
    },
    {
        "code": "public class Example101 {\n    public void execUserCommand(String cmd) throws IOException {\n        Runtime.getRuntime().exec(cmd);\n    }\n}",
        "expected": "VULNERABLE",
        "description": "Command injection vulnerability by executing user-supplied command",
    },
    {
        "code": 'public class Example102 {\n    public void queryUser(Connection conn, String id) throws SQLException {\n        PreparedStatement stmt = conn.prepareStatement("SELECT * FROM users WHERE id = ?");\n        stmt.setString(1, id);\n        stmt.executeQuery();\n    }\n}',
        "expected": "NOT_VULNERABLE",
        "description": "Secure SQL query using prepared statement",
    },
    {
        "code": 'public class Example103 {\n    public void queryUser(Connection conn, String id) throws SQLException {\n        Statement stmt = conn.createStatement();\n        stmt.execute("SELECT * FROM users WHERE id = " + id);\n    }\n}',
        "expected": "VULNERABLE",
        "description": "SQL injection vulnerability due to unvalidated concatenation",
    },
    {
        "code": 'public class Example104 {\n    public String hash(String password) throws Exception {\n        MessageDigest md = MessageDigest.getInstance("MD5");\n        return Base64.getEncoder().encodeToString(md.digest(password.getBytes()));\n    }\n}',
        "expected": "VULNERABLE",
        "description": "Weak hash algorithm MD5 used for password hashing",
    },
    {
        "code": 'public class Example105 {\n    public String hash(String password) throws Exception {\n        MessageDigest md = MessageDigest.getInstance("SHA-256");\n        return Base64.getEncoder().encodeToString(md.digest(password.getBytes()));\n    }\n}',
        "expected": "NOT_VULNERABLE",
        "description": "Secure hashing using SHA-256",
    },
    {
        "code": 'public class Example106 {\n    public void render(HttpServletRequest req, HttpServletResponse res) throws IOException {\n        String input = req.getParameter("q");\n        res.getWriter().write(input);\n    }\n}',
        "expected": "VULNERABLE",
        "description": "XSS vulnerability due to direct reflection of input",
    },
    {
        "code": 'public class Example107 {\n    public void render(HttpServletRequest req, HttpServletResponse res) throws IOException {\n        String input = req.getParameter("q");\n        String safe = org.apache.commons.text.StringEscapeUtils.escapeHtml4(input);\n        res.getWriter().write(safe);\n    }\n}',
        "expected": "NOT_VULNERABLE",
        "description": "Escaped output to prevent XSS",
    },
    {
        "code": 'public class Example108 {\n    public void openFile(String name) throws IOException {\n        File file = new File("uploads/" + name);\n        new FileInputStream(file);\n    }\n}',
        "expected": "VULNERABLE",
        "description": "Path traversal vulnerability by allowing arbitrary file name",
    },
    {
        "code": 'public class Example109 {\n    public void openFile(String name) throws IOException {\n        if (!name.matches("[a-zA-Z0-9]+\\\\.txt")) throw new IllegalArgumentException();\n        File file = new File("uploads/" + name);\n        new FileInputStream(file);\n    }\n}',
        "expected": "NOT_VULNERABLE",
        "description": "Validated file name prevents path traversal",
    },
    {
        "code": "public class Example110 {\n    public boolean isSame(String a, String b) {\n        return a.equals(b);\n    }\n}",
        "expected": "VULNERABLE",
        "description": "Potential timing attack due to standard equals comparison",
    },
    {
        "code": "public class Example111 {\n    public boolean isSame(String a, String b) {\n        return MessageDigest.isEqual(a.getBytes(), b.getBytes());\n    }\n}",
        "expected": "NOT_VULNERABLE",
        "description": "Safe constant-time comparison to avoid timing attack",
    },
    {
        "code": 'public class Example112 {\n    public void setSecurityHeaders(HttpServletResponse response) {\n        response.setHeader("Content-Security-Policy", "default-src \'self\'");\n        response.setHeader("X-Content-Type-Options", "nosniff");\n        response.setHeader("X-Frame-Options", "DENY");\n    }\n}',
        "expected": "NOT_VULNERABLE",
        "description": "Proper security headers set",
    },
    {
        "code": "public class Example113 {\n    public void setSecurityHeaders(HttpServletResponse response) {\n        // no headers set intentionally\n    }\n}",
        "expected": "VULNERABLE",
        "description": "Missing security headers exposing app to XSS/Clickjacking",
    },
    {
        "code": 'public class Example114 {\n    public void connect() {\n        String apiKey = "ABCD1234SECRET";\n        System.out.println(apiKey);\n    }\n}',
        "expected": "VULNERABLE",
        "description": "Hardcoded API key vulnerability",
    },
    {
        "code": 'public class Example115 {\n    public void connect() {\n        String apiKey = System.getenv("API_KEY");\n        System.out.println(apiKey);\n    }\n}',
        "expected": "NOT_VULNERABLE",
        "description": "API key from environment variable (safer practice)",
    },
    {
        "code": 'public class Test1 {\n    public static void main(String[] args) {\n        String apiKey = "ABCD1234";\n        System.out.println("Using API key: " + apiKey);\n    }\n}',
        "expected": "VULNERABLE",
        "description": "Hardcoded API key",
    },
    {
        "code": 'public class Test2 {\n    private final String DB_PASSWORD = "pass123";\n    public void connect() {\n        System.out.println("Connecting with password: " + DB_PASSWORD);\n    }\n}',
        "expected": "VULNERABLE",
        "description": "Hardcoded database password",
    },
    {
        "code": 'class Secrets {\n    static {\n        System.setProperty("secret", "mysecretpass");\n    }\n}',
        "expected": "VULNERABLE",
        "description": "Hardcoded secret via system property",
    },
    {
        "code": 'public class SafeToken {\n    public boolean validateToken(String input) {\n        String envToken = System.getenv("AUTH_TOKEN");\n        return input.equals(envToken);\n    }\n}',
        "expected": "NOT_VULNERABLE",
        "description": "Secure token comparison using environment variable",
    },
    {
        "code": 'String stored = System.getProperty("token");\nif (MessageDigest.isEqual(input.getBytes(), stored.getBytes())) {\n    grantAccess();\n}',
        "expected": "NOT_VULNERABLE",
        "description": "Constant-time secure string comparison",
    },
    {
        "code": 'public class UnsafeFile {\n    public void read(String filename) {\n        new FileInputStream("/data/" + filename);\n    }\n}',
        "expected": "VULNERABLE",
        "description": "Unvalidated file name allows path traversal",
    },
    {
        "code": 'public class SafeFile {\n    public void read(String filename) {\n        if (!filename.matches("[a-zA-Z0-9_]+\\.txt")) {\n            throw new IllegalArgumentException();\n        }\n        new FileInputStream("/safe_dir/" + filename);\n    }\n}',
        "expected": "NOT_VULNERABLE",
        "description": "Validated file name prevents path traversal",
    },
    {
        "code": "public boolean verifyPin(String input, String correct) {\n    return input.equals(correct);\n}",
        "expected": "VULNERABLE",
        "description": "Timing attack possible due to string equality",
    },
    {
        "code": "if (userHash.equals(dbHash)) { allow(); }",
        "expected": "VULNERABLE",
        "description": "Direct string comparison vulnerable to timing attacks",
    },
    {
        "code": 'public String hash(String pass) {\n    MessageDigest md = MessageDigest.getInstance("MD5");\n    return Base64.getEncoder().encodeToString(md.digest(pass.getBytes()));\n}',
        "expected": "VULNERABLE",
        "description": "Using weak hashing algorithm MD5",
    },
    {
        "code": "SecureRandom random = new SecureRandom();\nbyte[] nonce = new byte[16];\nrandom.nextBytes(nonce);",
        "expected": "NOT_VULNERABLE",
        "description": "Use of cryptographically strong random generator",
    },
    {
        "code": 'Runtime.getRuntime().exec("ping " + userInput);',
        "expected": "VULNERABLE",
        "description": "User input directly used in command execution",
    },
    {
        "code": 'ProcessBuilder pb = new ProcessBuilder("sh", "-c", "echo " + userCmd);\npb.start();',
        "expected": "VULNERABLE",
        "description": "Command injection via shell execution",
    },
    {
        "code": 'Process p = Runtime.getRuntime().exec(new String[]{"ls", "-la"});',
        "expected": "NOT_VULNERABLE",
        "description": "Safe use of hardcoded command",
    },
    {
        "code": 'Statement stmt = connection.createStatement();\nResultSet rs = stmt.executeQuery("SELECT * FROM users WHERE id = " + id);',
        "expected": "VULNERABLE",
        "description": "Concatenating user input into SQL query",
    },
    {
        "code": 'PreparedStatement ps = connection.prepareStatement("SELECT * FROM users WHERE id = ?");\nps.setString(1, id);\nps.executeQuery();',
        "expected": "NOT_VULNERABLE",
        "description": "Parameterized query prevents SQL injection",
    },
    {
        "code": "public void setHeaders(HttpServletResponse response) {\n    // No security headers set\n}",
        "expected": "VULNERABLE",
        "description": "Missing security headers allowing XSS and other attacks",
    },
    {
        "code": 'response.setHeader("Content-Security-Policy", "default-src \'self\'");',
        "expected": "NOT_VULNERABLE",
        "description": "Security header set correctly",
    },
    {
        "code": 'if (input != null && input.equals("admin")) grantAdmin();',
        "expected": "VULNERABLE",
        "description": "Hardcoded credential check",
    },
    {
        "code": 'response.setHeader("X-Frame-Options", "DENY");',
        "expected": "NOT_VULNERABLE",
        "description": "Security header prevents clickjacking",
    },
    {
        "code": 'try (InputStream is = getClass().getResourceAsStream("/config/secrets.properties")) {}',
        "expected": "NOT_VULNERABLE",
        "description": "Reading secrets from resource stream securely",
    },
    {
        "code": 'new File(System.getProperty("user.home") + "/.app/config.txt");',
        "expected": "NOT_VULNERABLE",
        "description": "Safe file access using system properties",
    },
    {
        "code": "return password.hashCode() == storedHash;",
        "expected": "VULNERABLE",
        "description": "Insecure password comparison using hash code",
    },
    {
        "code": "public class Example {\n    public static void main(String[] args) {\n        byte[] buffer = new byte[10];\n        System.in.read(buffer);\n    }\n}",
        "vulnerability_type": "Buffer Overflow",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    public void transferMoney(Account from, Account to, double amount) {\n        if (from.getBalance() >= amount) {\n            from.withdraw(amount);\n            to.deposit(amount);\n        }\n    }\n}",
        "vulnerability_type": "Race Condition",
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public String encrypt(String data) throws Exception {\n        Cipher cipher = Cipher.getInstance("DES");\n        cipher.init(Cipher.ENCRYPT_MODE, new SecretKeySpec("key12345".getBytes(), "DES"));\n        byte[] encryptedData = cipher.doFinal(data.getBytes());\n        return Base64.getEncoder().encodeToString(encryptedData);\n    }\n}',
        "vulnerability_type": "Weak Encryption",
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public boolean authenticate(String username, String password) {\n        return username.equals("admin") && password.equals("password");\n    }\n}',
        "vulnerability_type": "Broken Authentication",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    public void configureSecurity(SecurityConfig config) {\n        config.disableCsrfProtection();\n        config.allowAllOrigins();\n    }\n}",
        "vulnerability_type": "Security Misconfiguration",
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void sendUserData(HttpServletResponse response, User user) throws IOException {\n        response.getWriter().write("User data: " + user.toString());\n    }\n}',
        "vulnerability_type": "Sensitive Data Exposure",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    public void processPayment(Payment payment) {\n        // Process payment without logging\n    }\n}",
        "vulnerability_type": "Insufficient Logging",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    public void fetchUrl(String url) throws Exception {\n        URL u = new URL(url);\n        HttpURLConnection connection = (HttpURLConnection) u.openConnection();\n        connection.getInputStream();\n    }\n}",
        "vulnerability_type": "SSRF",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    public void parseXML(String xml) throws Exception {\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        builder.parse(new InputSource(new StringReader(xml)));\n    }\n}",
        "vulnerability_type": "XXE",
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void deleteUser(HttpServletRequest request) {\n        String userId = request.getParameter("userId");\n        userDao.delete(userId);\n    }\n}',
        "vulnerability_type": "Broken Access Control",
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void processAPIRequest(HttpServletRequest request) {\n        String apiKey = request.getHeader("API-Key");\n        if (apiKey.equals("12345")) {\n            // Process request\n        }\n    }\n}',
        "vulnerability_type": "Insecure API Usage",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    public void readFile(String filePath) throws Exception {\n        File file = new File(filePath);\n        FileInputStream fis = new FileInputStream(file);\n    }\n}",
        "vulnerability_type": "Directory Traversal",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    public void processData(byte[] data) {\n        // Process data without bounds checking\n        System.out.println(new String(data));\n    }\n}",
        "vulnerability_type": "Heap Inspection",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    private static List<Object> cache = new ArrayList<>();\n    public void addToCache(Object obj) {\n        cache.add(obj);\n    }\n}",
        "vulnerability_type": "Memory Leak",
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void processLargeInput(String input) {\n        // Process large input without size limits\n        String[] parts = input.split(",");\n        for (String part : parts) {\n            System.out.println(part);\n        }\n    }\n}',
        "vulnerability_type": "DoS",
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void uploadFile(HttpServletRequest request) throws Exception {\n        Part filePart = request.getPart("file");\n        String fileName = filePart.getSubmittedFileName();\n        filePart.write("uploads/" + fileName);\n    }\n}',
        "vulnerability_type": "Insecure File Upload",
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void sendDataOverHttp(String data) throws Exception {\n        URL url = new URL("http://example.com/api");\n        HttpURLConnection connection = (HttpURLConnection) url.openConnection();\n        connection.setRequestMethod("POST");\n        connection.getOutputStream().write(data.getBytes());\n    }\n}',
        "vulnerability_type": "Insecure Communication",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    public void validateCertificate(Certificate cert) {\n        // No validation logic\n    }\n}",
        "vulnerability_type": "Improper Certificate Validation",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    public void configureServer() {\n        Server server = new Server(8080);\n        server.setDefaultServlet(new DefaultServlet());\n        // No security configurations\n    }\n}",
        "vulnerability_type": "Insecure Default Configurations",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    public void processRequest(HttpServletRequest request) {\n        try {\n            // Process request\n        } catch (Exception e) {\n            e.printStackTrace();\n        }\n    }\n}",
        "vulnerability_type": "Improper Exception Handling",
        "expected": "VULNERABLE",
    },
    {
        "code": "public class Example {\n    public static void main(String[] args) throws IOException {\n        byte[] buffer = new byte[1024];\n        int bytesRead = System.in.read(buffer, 0, buffer.length);\n        if (bytesRead > 0) {\n            System.out.write(buffer, 0, bytesRead);\n        }\n    }\n}",
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": "public class Example {\n    public synchronized void transferMoney(Account from, Account to, double amount) {\n        if (from.getBalance() >= amount) {\n            from.withdraw(amount);\n            to.deposit(amount);\n        }\n    }\n}",
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'import javax.crypto.Cipher;\nimport javax.crypto.spec.SecretKeySpec;\nimport java.util.Base64;\n\npublic class Example {\n    public String encrypt(String data) throws Exception {\n        Cipher cipher = Cipher.getInstance("AES/GCM/NoPadding");\n        SecretKeySpec keySpec = new SecretKeySpec("1234567890123456".getBytes(), "AES");\n        cipher.init(Cipher.ENCRYPT_MODE, keySpec);\n        byte[] encryptedData = cipher.doFinal(data.getBytes());\n        return Base64.getEncoder().encodeToString(encryptedData);\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public boolean authenticate(String username, String password) {\n        // Assume getHashedPassword retrieves the hashed password from a secure store\n        String storedHash = getHashedPassword(username);\n        String inputHash = hashPassword(password);\n        return secureCompare(storedHash, inputHash);\n    }\n\n    private String hashPassword(String password) {\n        // Implement secure hashing\n        return "hashed_password";\n    }\n\n    private boolean secureCompare(String a, String b) {\n        // Implement constant-time comparison\n        if (a.length() != b.length()) return false;\n        int result = 0;\n        for (int i = 0; i < a.length(); i++) {\n            result |= a.charAt(i) ^ b.charAt(i);\n        }\n        return result == 0;\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void configureSecurity(SecurityConfig config) {\n        config.enableCsrfProtection();\n        config.setAllowedOrigins(new String[]{"https://trusted.com"});\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void sendUserData(HttpServletResponse response, User user) throws IOException {\n        response.getWriter().write("User data: " + sanitize(user.toString()));\n    }\n\n    private String sanitize(String input) {\n        // Implement data sanitization\n        return input.replaceAll("[^a-zA-Z0-9]", "");\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    private static final Logger logger = Logger.getLogger(Example.class.getName());\n\n    public void processPayment(Payment payment) {\n        logger.info("Processing payment for user: " + payment.getUserId());\n        // Process payment\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void fetchUrl(String url) throws Exception {\n        URL u = new URL(url);\n        if (!u.getHost().equals("trusted.com")) {\n            throw new SecurityException("Invalid URL");\n        }\n        HttpURLConnection connection = (HttpURLConnection) u.openConnection();\n        connection.getInputStream();\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void parseXML(String xml) throws Exception {\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        factory.setFeature("http://apache.org/xml/features/disallow-doctype-decl", true);\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        builder.parse(new InputSource(new StringReader(xml)));\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void deleteUser(HttpServletRequest request) {\n        String userId = request.getParameter("userId");\n        String currentUserId = request.getSession().getAttribute("userId");\n        if (!userId.equals(currentUserId) && !isAdmin(request)) {\n            throw new SecurityException("Unauthorized access");\n        }\n        userDao.delete(userId);\n    }\n\n    private boolean isAdmin(HttpServletRequest request) {\n        // Check if user is admin\n        return false;\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void processAPIRequest(HttpServletRequest request) {\n        String apiKey = request.getHeader("API-Key");\n        if (!validateApiKey(apiKey)) {\n            throw new SecurityException("Invalid API key");\n        }\n        // Process request\n    }\n\n    private boolean validateApiKey(String apiKey) {\n        // Implement secure API key validation\n        return apiKey != null && apiKey.equals(getSecureApiKey());\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void readFile(String filePath) throws Exception {\n        File file = new File("safe_directory/" + filePath);\n        if (!file.getCanonicalPath().startsWith(new File("safe_directory").getCanonicalPath())) {\n            throw new SecurityException("Invalid file path");\n        }\n        FileInputStream fis = new FileInputStream(file);\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void processData(byte[] data) {\n        if (data.length > 1024) {\n            throw new IllegalArgumentException("Data too large");\n        }\n        System.out.println(new String(data, 0, Math.min(data.length, 1024)));\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": "public class Example {\n    private static final Map<String, Object> cache = new HashMap<>();\n    public void addToCache(String key, Object obj) {\n        if (cache.size() > 1000) {\n            cache.clear();\n        }\n        cache.put(key, obj);\n    }\n}",
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void processLargeInput(String input) {\n        if (input.length() > 1000) {\n            throw new IllegalArgumentException("Input too large");\n        }\n        String[] parts = input.split(",", 100);\n        for (String part : parts) {\n            System.out.println(part);\n        }\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void uploadFile(HttpServletRequest request) throws Exception {\n        Part filePart = request.getPart("file");\n        String fileName = filePart.getSubmittedFileName();\n        if (!fileName.endsWith(".txt")) {\n            throw new IllegalArgumentException("Only .txt files are allowed");\n        }\n        filePart.write("uploads/" + sanitizeFileName(fileName));\n    }\n\n    private String sanitizeFileName(String fileName) {\n        return fileName.replaceAll("[^a-zA-Z0-9.]", "_");\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void sendDataOverHttps(String data) throws Exception {\n        URL url = new URL("https://example.com/api");\n        HttpsURLConnection connection = (HttpsURLConnection) url.openConnection();\n        connection.setRequestMethod("POST");\n        connection.getOutputStream().write(data.getBytes());\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void validateCertificate(X509Certificate cert) throws CertificateException {\n        try {\n            cert.checkValidity();\n            // Additional validation checks\n        } catch (Exception e) {\n            throw new CertificateException("Invalid certificate");\n        }\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": "public class Example {\n    public void configureServer() {\n        Server server = new Server(8443);\n        server.setDefaultServlet(new SecureServlet());\n        server.setSecurityHandler(new SecurityHandler());\n    }\n}",
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example {\n    public void processRequest(HttpServletRequest request) {\n        try {\n            // Process request\n        } catch (Exception e) {\n            logger.error("Error processing request", e);\n            throw new ServletException("An error occurred", e);\n        }\n    }\n}',
        "vulnerability_type": "safe",
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'String dbPassword = "p@ssw0rd123"; // Hardcoded DB password',
        "expected": "VULNERABLE",
        "description": "Hardcoded credentials in plaintext",
    },
    {
        "code": 'final String API_KEY = System.getenv("SECURE_API_KEY");',
        "expected": "NOT_VULNERABLE",
        "description": "Secure credential loading from environment variables",
    },
    {
        "code": 'String query = "DELETE FROM users WHERE id = " + userInput;',
        "expected": "VULNERABLE",
        "description": "SQL Injection in DELETE statement",
    },
    {
        "code": 'String query = "SELECT * FROM products WHERE category = ?";\nPreparedStatement stmt = conn.prepareStatement(query);\nstmt.setString(1, userInput);',
        "expected": "NOT_VULNERABLE",
        "description": "Parameterized query prevents SQLi",
    },
    {
        "code": 'Runtime.getRuntime().exec("sh -c \'" + userInput + "\'");',
        "expected": "VULNERABLE",
        "description": "Command injection via shell concatenation",
    },
    {
        "code": 'String[] cmd = {"echo", userInput};\nRuntime.getRuntime().exec(cmd);',
        "expected": "VULNERABLE",
        "description": "Unsanitized user input in command array",
    },
    {
        "code": 'response.getWriter().print("<div>" + userComment + "</div>");',
        "expected": "VULNERABLE",
        "description": "Direct user input in HTML without escaping",
    },
    {
        "code": 'response.getWriter().print("<div>" + Encode.forHtml(userComment) + "</div>");',
        "expected": "NOT_VULNERABLE",
        "description": "Proper HTML output encoding",
    },
    {
        "code": "ObjectInputStream ois = new ObjectInputStream(request.getInputStream());\nObject obj = ois.readObject();",
        "expected": "VULNERABLE",
        "description": "Untrusted deserialization without validation",
    },
    {
        "code": "ObjectInputStream ois = new SafeObjectInputStream(request.getInputStream(), SafeClass.class);",
        "expected": "NOT_VULNERABLE",
        "description": "Deserialization with allowlist validation",
    },
    {
        "code": 'File file = new File("/var/www/uploads/" + userSuppliedFilename);',
        "expected": "VULNERABLE",
        "description": "Potential path traversal via filename",
    },
    {
        "code": 'String safeName = userSuppliedFilename.replaceAll("[^a-zA-Z0-9.-]", "_");\nFile file = new File("/var/www/uploads/", safeName);',
        "expected": "NOT_VULNERABLE",
        "description": "Filename sanitization prevents traversal",
    },
    {
        "code": 'Cipher cipher = Cipher.getInstance("DES/ECB/PKCS5Padding");',
        "expected": "VULNERABLE",
        "description": "Weak cipher algorithm (DES) and mode (ECB)",
    },
    {
        "code": 'Cipher cipher = Cipher.getInstance("AES/GCM/NoPadding");\ncipher.init(Cipher.ENCRYPT_MODE, key, new GCMParameterSpec(128, iv));',
        "expected": "NOT_VULNERABLE",
        "description": "Secure AES-GCM configuration",
    },
    {
        "code": "URL url = new URL(userSuppliedUrl);\nHttpURLConnection conn = (HttpURLConnection) url.openConnection();",
        "expected": "VULNERABLE",
        "description": "Unrestricted URL fetching enables SSRF",
    },
    {
        "code": 'if (userSuppliedUrl.startsWith("https://trusted-api.com/")) {\n  URL url = new URL(userSuppliedUrl);\n  HttpURLConnection conn = (HttpURLConnection) url.openConnection();\n}',
        "expected": "NOT_VULNERABLE",
        "description": "URL allowlist prevents SSRF",
    },
    {
        "code": "int sessionToken = new Random().nextInt();",
        "expected": "VULNERABLE",
        "description": "Insecure random number for security-sensitive value",
    },
    {
        "code": "int sessionToken = new SecureRandom().nextInt();",
        "expected": "NOT_VULNERABLE",
        "description": "Cryptographically secure random number",
    },
]


In [ ]:
# @title
import json
import json
def evaluate_model(test_cases):
    results = []
    correct = 0

    for i, test_case in enumerate(test_cases):
        code = test_case["code"]
        expected = test_case["expected"]

        print(f"\nEvaluating test case {i+1}...")
        # print("Code snippet:")
        # print(code[:200] + "..." if len(code) > 200 else code)

        # Get model prediction
        try:
            prediction = classify_code(code)
            print("Model output:", prediction)

            # Extract classification from prediction
            if "Classification: VULNERABLE" in prediction:
                predicted_class = "VULNERABLE"
            elif "Classification: NOT_VULNERABLE" in prediction:
                predicted_class = "NOT_VULNERABLE"
            else:
                predicted_class = "UNKNOWN"

            # Check if prediction matches expected
            is_correct = predicted_class == expected
            if is_correct:
                correct += 1

            results.append({
                "test_case": i+1,
                "expected": expected,
                "predicted": predicted_class,
                "correct": is_correct,
                "model_output": prediction
            })

        except Exception as e:
            print(f"Error processing test case {i+1}: {str(e)}")
            results.append({
                "test_case": i+1,
                "error": str(e)
            })

    # Calculate accuracy
    accuracy = correct / len(test_cases) * 100

    # Print summary
    print("\nEvaluation Summary:")
    print(f"Total test cases: {len(test_cases)}")
    print(f"Correct predictions: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")

    # Save detailed results
    with open("evaluation_results.json", "w") as f:
        json.dump({
            "summary": {
                "total_test_cases": len(test_cases),
                "correct_predictions": correct,
                "accuracy": accuracy
            },
            "detailed_results": results
        }, f, indent=2)

    return results

In [ ]:
# @title
def classify_code(code):
    messages = [
        {
            "role": "system",
            "content": "You are a code vulnerability classifier. Analyze the provided Java code and classify it as either 'VULNERABLE' or 'NOT VULNERABLE'. If vulnerable, specify the exact vulnerability type from the dataset. Respond strictly in this format: 'Classification: <VULNERABLE/NOT VULNERABLE>. Vulnerability Type: <type>'. Do not include reasoning.",
        },
        {"role": "user", "content": code},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    # Generate output
    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=256
    )

    # Decode and clean the output
    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Extract just the model's response (after the last user message)
    last_user_msg = messages[-1]["content"]
    response = output_text.split(last_user_msg)[-1].strip()

    return response


In [ ]:
import json
evaluation_results = evaluate_model(testcases)

In [ ]:
# @title
def classify_code(code):
    messages = [
        {
            "role": "system",
            "content": "You are a code vulnerability classifier. Analyze the provided Java code and classify it as either 'VULNERABLE' or 'NOT VULNERABLE'. If vulnerable, specify the exact vulnerability type from the dataset. Respond strictly in this format: 'Classification: <VULNERABLE/NOT VULNERABLE>. Vulnerability Type: <type>'. Do not include reasoning.",
        },
        {"role": "user", "content": code},
    ]

    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs["input_ids"],
        attention_mask=inputs["attention_mask"],
        max_new_tokens=256
    )

    output_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    last_user_msg = messages[-1]["content"]
    response = output_text.split(last_user_msg)[-1].strip()

    return response


In [ ]:
# @title
import json
def evaluate_model(llm_name,test_cases):
    results = []
    correct = 0

    for i, test_case in enumerate(test_cases):
        code = test_case["code"]
        expected = test_case["expected"]
        # print(f"\nEvaluating test case {i+1}...")
        # print("Code snippet:")
        # print(code[:200] + "..." if len(code) > 200 else code)
        try:
            prediction = classify_code(code)
            # print("Model output:", prediction)
            if "Classification: VULNERABLE" in prediction:
                predicted_class = "VULNERABLE"
            elif "Classification: NOT_VULNERABLE" in prediction:
                predicted_class = "NOT_VULNERABLE"
            else:
                predicted_class = "UNKNOWN"
            is_correct = predicted_class == expected
            if is_correct:
                correct += 1

            results.append({
                "test_case": i+1,
                "expected": expected,
                "predicted": predicted_class,
                "correct": is_correct,
                "model_output": prediction
            })

        except Exception as e:
            print(f"Error processing test case {i+1}: {str(e)}")
            results.append({
                "test_case": i+1,
                "error": str(e)
            })

    accuracy = correct / len(test_cases) * 100
    print(f"\n{llm_name} Testcases Evaluation Summary:")
    print(f"Total test cases: {len(test_cases)}")
    print(f"Correct predictions: {correct}")
    print(f"Accuracy: {accuracy:.2f}%")
    output_file_path=llm_name+"_evaluation_results.json"
    with open(output_file_path, "w") as f:
        json.dump({
            "summary": {
                "total_test_cases": len(test_cases),
                "correct_predictions": correct,
                "accuracy": accuracy
            },
            "detailed_results": results
        }, f, indent=2)

    return results


In [ ]:
# @title
deepseek_test=[
    {
        "code": 'public class Example1 {\n    public static void main(String[] args) {\n        String username = "admin";\n        String password = "admin123";\n        if (args[0].equals(username) && args[1].equals(password)) {\n            System.out.println("Access granted");\n        }\n    }\n}',
        "expected": "VULNERABLE"
    },
    {
        "code": 'public class Example2 {\n    public static void main(String[] args) {\n        String query = "SELECT * FROM users WHERE username = \'" + args[0] + "\'";\n        Statement stmt = connection.createStatement();\n        ResultSet rs = stmt.executeQuery(query);\n    }\n}',
        "expected": "VULNERABLE"
    },
    {
        "code": 'public class Example3 {\n    public static void main(String[] args) {\n        String input = args[0];\n        System.out.println("<div>" + input + "</div>");\n    }\n}',
        "expected": "VULNERABLE"
    },
    {
        "code": 'public class Example4 {\n    public static void main(String[] args) throws Exception {\n        String fileName = "../../etc/passwd";\n        File file = new File("uploads/" + fileName);\n        FileInputStream fis = new FileInputStream(file);\n    }\n}',
        "expected": "VULNERABLE"
    },
    {
        "code": 'public class Example5 {\n    public static void main(String[] args) {\n        String secretKey = "A1B2C3D4E5F6";\n        Cipher cipher = Cipher.getInstance("AES");\n        cipher.init(Cipher.ENCRYPT_MODE, new SecretKeySpec(secretKey.getBytes(), "AES"));\n    }\n}',
        "expected": "VULNERABLE"
    },
    {
        "code": 'public class Example6 {\n    public static void main(String[] args) {\n        String username = System.getenv("DB_USER");\n        String password = System.getenv("DB_PASS");\n        String query = "SELECT * FROM users WHERE username = ?";\n        PreparedStatement stmt = connection.prepareStatement(query);\n        stmt.setString(1, args[0]);\n        ResultSet rs = stmt.executeQuery();\n    }\n}',
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": 'public class Example7 {\n    public static void main(String[] args) {\n        String input = args[0];\n        String safeInput = input.replaceAll("<", "&lt;").replaceAll(">", "&gt;");\n        System.out.println("<div>" + safeInput + "</div>");\n    }\n}',
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": 'public class Example8 {\n    public static void main(String[] args) throws Exception {\n        String fileName = args[0];\n        Path safePath = Paths.get("uploads/").resolve(Paths.get(fileName)).normalize();\n        if (!safePath.startsWith("uploads/")) {\n            throw new IllegalArgumentException("Invalid file path");\n        }\n        FileInputStream fis = new FileInputStream(safePath.toFile());\n    }\n}',
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": 'public class Example9 {\n    public static void main(String[] args) {\n        String algorithm = "AES/CBC/PKCS5Padding";\n        KeyGenerator keyGen = KeyGenerator.getInstance("AES");\n        keyGen.init(256);\n        SecretKey secretKey = keyGen.generateKey();\n        Cipher cipher = Cipher.getInstance(algorithm);\n        cipher.init(Cipher.ENCRYPT_MODE, secretKey);\n    }\n}',
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": 'public class Example10 {\n    public static void main(String[] args) {\n        String userInput = args[0];\n        if (userInput.matches("[a-zA-Z0-9]+")) {\n            String query = "SELECT * FROM users WHERE username = \'" + userInput + "\'";\n            Statement stmt = connection.createStatement();\n            ResultSet rs = stmt.executeQuery(query);\n        }\n    }\n}',
        "expected": "VULNERABLE"
    }
]

gemini_testcases=[
    {
        "code": "import java.sql.Connection;\nimport java.sql.DriverManager;\nimport java.sql.Statement;\n\npublic class SQLInjectionVulnerable {\n    public void queryDatabase(String userInput) throws Exception {\n        Connection con = DriverManager.getConnection(\"jdbc:mysql://localhost:3306/db\", \"user\", \"password\");\n        Statement stmt = con.createStatement();\n        String query = \"SELECT * FROM users WHERE username = '\" + userInput + \"'\";\n        stmt.executeQuery(query);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.sql.Connection;\nimport java.sql.DriverManager;\nimport java.sql.PreparedStatement;\n\npublic class SQLInjectionNotVulnerable {\n    public void queryDatabase(String userInput) throws Exception {\n        Connection con = DriverManager.getConnection(\"jdbc:mysql://localhost:3306/db\", \"user\", \"password\");\n        String query = \"SELECT * FROM users WHERE username = ?\";\n        PreparedStatement pstmt = con.prepareStatement(query);\n        pstmt.setString(1, userInput);\n        pstmt.executeQuery();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.io.File;\n\npublic class PathTraversalVulnerable {\n    public void readFile(String fileName) {\n        File file = new File(\"/var/www/html/\" + fileName);\n        // ... code to read file\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.File;\n\npublic class PathTraversalNotVulnerable {\n    public void readFile(String fileName) {\n        File file = new File(fileName);\n        if (!file.getAbsolutePath().startsWith(\"/var/www/html/\")) {\n            throw new SecurityException(\"Access Denied\");\n        }\n        // ... code to read file\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import javax.servlet.http.HttpServletRequest;\nimport javax.servlet.http.HttpServletResponse;\nimport java.io.IOException;\n\npublic class XSSVulnerable {\n    protected void doGet(HttpServletRequest request, HttpServletResponse response) throws IOException {\n        String comment = request.getParameter(\"comment\");\n        response.getWriter().println(\"<html><body>\" + comment + \"</body></html>\");\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import javax.servlet.http.HttpServletRequest;\nimport javax.servlet.http.HttpServletResponse;\nimport java.io.IOException;\nimport org.owasp.encoder.Encode;\n\npublic class XSSNotVulnerable {\n    protected void doGet(HttpServletRequest request, HttpServletResponse response) throws IOException {\n        String comment = request.getParameter(\"comment\");\n        response.getWriter().println(\"<html><body>\" + Encode.forHtml(comment) + \"</body></html>\");\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class HardcodedPasswordVulnerable {\n    public void connectToDb() {\n        String dbPassword = \"mySuperSecretPassword123\";\n        // ... code to connect to database\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.util.Properties;\nimport java.io.FileInputStream;\nimport java.io.IOException;\n\npublic class HardcodedPasswordNotVulnerable {\n    public String getDbPassword() throws IOException {\n        Properties props = new Properties();\n        FileInputStream fis = new FileInputStream(\"config.properties\");\n        props.load(fis);\n        return props.getProperty(\"db.password\");\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import javax.xml.parsers.DocumentBuilder;\nimport javax.xml.parsers.DocumentBuilderFactory;\nimport org.xml.sax.InputSource;\nimport java.io.StringReader;\n\npublic class XXEVulnerable {\n    public void parseXML(String xml) throws Exception {\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        builder.parse(new InputSource(new StringReader(xml)));\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import javax.xml.parsers.DocumentBuilder;\nimport javax.xml.parsers.DocumentBuilderFactory;\nimport org.xml.sax.InputSource;\nimport java.io.StringReader;\n\npublic class XXENotVulnerable {\n    public void parseXML(String xml) throws Exception {\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        factory.setFeature(\"http://apache.org/xml/features/disallow-doctype-decl\", true);\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        builder.parse(new InputSource(new StringReader(xml)));\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.io.*;\n\npublic class DeserializationVulnerable {\n    public void process(InputStream input) throws IOException, ClassNotFoundException {\n        ObjectInputStream ois = new ObjectInputStream(input);\n        Object obj = ois.readObject();\n        // ...\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.*;\n\npublic class DeserializationNotVulnerable {\n    // Using a safe serialization format like JSON with a library like Jackson\n    // is preferred over native Java serialization.\n    public void process(String jsonInput) {\n        // Use a library like Jackson or Gson to parse JSON safely.\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.util.Random;\n\npublic class InsecureRandomVulnerable {\n    public int generateToken() {\n        Random rand = new Random();\n        return rand.nextInt();\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.security.SecureRandom;\n\npublic class InsecureRandomNotVulnerable {\n    public int generateToken() {\n        SecureRandom rand = new SecureRandom();\n        return rand.nextInt();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class CommandInjectionVulnerable {\n    public void runCommand(String userInput) throws java.io.IOException {\n        Runtime.getRuntime().exec(\"cmd /c dir \" + userInput);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class CommandInjectionNotVulnerable {\n    public void runCommand(String userInput) throws java.io.IOException {\n        String[] command = {\"cmd\", \"/c\", \"dir\", userInput};\n        Runtime.getRuntime().exec(command);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    }
]

claude_testcases=[
    {
        "code": "public class SQLInjectionVulnerable {\n    public void getUserData(String userId) {\n        String query = \"SELECT * FROM users WHERE id = '\" + userId + \"'\";\n        // Execute query without preparation\n        executeQuery(query);\n    }\n    private void executeQuery(String query) { /* implementation */ }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SQLInjectionSecure {\n    public void getUserData(String userId) {\n        String query = \"SELECT * FROM users WHERE id = ?\";\n        PreparedStatement stmt = connection.prepareStatement(query);\n        stmt.setString(1, userId);\n        stmt.executeQuery();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class PathTraversalVulnerable {\n    public void readFile(String fileName) throws IOException {\n        File file = new File(\"/uploads/\" + fileName);\n        FileInputStream fis = new FileInputStream(file);\n        // Process file\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class PathTraversalSecure {\n    public void readFile(String fileName) throws IOException {\n        if (fileName.contains(\"..\") || fileName.contains(\"/\") || fileName.contains(\"\\\\\")) {\n            throw new SecurityException(\"Invalid file name\");\n        }\n        File file = new File(\"/uploads/\" + fileName);\n        FileInputStream fis = new FileInputStream(file);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class XSSVulnerable {\n    public void displayUserInput(String userInput) {\n        String html = \"<div>\" + userInput + \"</div>\";\n        response.getWriter().println(html);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class XSSSecure {\n    public void displayUserInput(String userInput) {\n        String sanitized = StringEscapeUtils.escapeHtml4(userInput);\n        String html = \"<div>\" + sanitized + \"</div>\";\n        response.getWriter().println(html);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class HardcodedCredentials {\n    private static final String DB_PASSWORD = \"admin123\";\n    private static final String API_KEY = \"sk-1234567890abcdef\";\n    \n    public void connect() {\n        Connection conn = DriverManager.getConnection(\"jdbc:mysql://localhost/db\", \"admin\", DB_PASSWORD);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SecureCredentials {\n    private String dbPassword;\n    private String apiKey;\n    \n    public SecureCredentials() {\n        this.dbPassword = System.getenv(\"DB_PASSWORD\");\n        this.apiKey = System.getenv(\"API_KEY\");\n    }\n    \n    public void connect() {\n        Connection conn = DriverManager.getConnection(\"jdbc:mysql://localhost/db\", \"admin\", dbPassword);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class WeakCrypto {\n    public String encryptData(String data) throws Exception {\n        Cipher cipher = Cipher.getInstance(\"DES\");\n        SecretKeySpec key = new SecretKeySpec(\"12345678\".getBytes(), \"DES\");\n        cipher.init(Cipher.ENCRYPT_MODE, key);\n        return Base64.getEncoder().encodeToString(cipher.doFinal(data.getBytes()));\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class StrongCrypto {\n    public String encryptData(String data) throws Exception {\n        Cipher cipher = Cipher.getInstance(\"AES/GCM/NoPadding\");\n        SecretKeySpec key = new SecretKeySpec(generateSecureKey(), \"AES\");\n        cipher.init(Cipher.ENCRYPT_MODE, key);\n        return Base64.getEncoder().encodeToString(cipher.doFinal(data.getBytes()));\n    }\n    \n    private byte[] generateSecureKey() {\n        SecureRandom random = new SecureRandom();\n        byte[] key = new byte[32];\n        random.nextBytes(key);\n        return key;\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class InsecureRandom {\n    public String generateToken() {\n        Random random = new Random();\n        return String.valueOf(random.nextLong());\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SecureRandom {\n    public String generateToken() {\n        java.security.SecureRandom random = new java.security.SecureRandom();\n        byte[] bytes = new byte[32];\n        random.nextBytes(bytes);\n        return Base64.getEncoder().encodeToString(bytes);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class CommandInjectionVulnerable {\n    public void executeCommand(String userInput) throws IOException {\n        String command = \"ls -la \" + userInput;\n        Runtime.getRuntime().exec(command);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class CommandInjectionSecure {\n    public void executeCommand(String userInput) throws IOException {\n        if (!userInput.matches(\"^[a-zA-Z0-9_-]+$\")) {\n            throw new IllegalArgumentException(\"Invalid input\");\n        }\n        String[] command = {\"ls\", \"-la\", userInput};\n        Runtime.getRuntime().exec(command);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class LDAPInjectionVulnerable {\n    public void searchUser(String username) throws NamingException {\n        String filter = \"(uid=\" + username + \")\";\n        DirContext ctx = new InitialDirContext();\n        ctx.search(\"ou=users,dc=example,dc=com\", filter, new SearchControls());\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class LDAPInjectionSecure {\n    public void searchUser(String username) throws NamingException {\n        String escapedUsername = username.replace(\"\\\\\", \"\\\\\\\\\").replace(\"*\", \"\\\\*\").replace(\"(\", \"\\\\(\").replace(\")\", \"\\\\)\");\n        String filter = \"(uid=\" + escapedUsername + \")\";\n        DirContext ctx = new InitialDirContext();\n        ctx.search(\"ou=users,dc=example,dc=com\", filter, new SearchControls());\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class TrustAllCertificates {\n    public void makeHttpsRequest() throws Exception {\n        TrustManager[] trustAllCerts = new TrustManager[] {\n            new X509TrustManager() {\n                public X509Certificate[] getAcceptedIssuers() { return null; }\n                public void checkClientTrusted(X509Certificate[] certs, String authType) { }\n                public void checkServerTrusted(X509Certificate[] certs, String authType) { }\n            }\n        };\n        SSLContext sc = SSLContext.getInstance(\"SSL\");\n        sc.init(null, trustAllCerts, new java.security.SecureRandom());\n        HttpsURLConnection.setDefaultSSLSocketFactory(sc.getSocketFactory());\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class ProperSSLValidation {\n    public void makeHttpsRequest() throws Exception {\n        SSLContext sslContext = SSLContext.getInstance(\"TLS\");\n        sslContext.init(null, null, null); // Use default trust managers\n        HttpsURLConnection connection = (HttpsURLConnection) url.openConnection();\n        connection.setSSLSocketFactory(sslContext.getSocketFactory());\n        connection.connect();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class XMLExternalEntity {\n    public void parseXML(String xmlContent) throws Exception {\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        Document doc = builder.parse(new ByteArrayInputStream(xmlContent.getBytes()));\n        // Process document\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SecureXMLParsing {\n    public void parseXML(String xmlContent) throws Exception {\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        factory.setFeature(\"http://apache.org/xml/features/disallow-doctype-decl\", true);\n        factory.setFeature(\"http://xml.org/sax/features/external-general-entities\", false);\n        factory.setFeature(\"http://xml.org/sax/features/external-parameter-entities\", false);\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        Document doc = builder.parse(new ByteArrayInputStream(xmlContent.getBytes()));\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class InsecureDeserialization {\n    public Object deserializeObject(byte[] data) throws Exception {\n        ObjectInputStream ois = new ObjectInputStream(new ByteArrayInputStream(data));\n        return ois.readObject();\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SecureDeserialization {\n    public Object deserializeObject(byte[] data) throws Exception {\n        ObjectInputStream ois = new ObjectInputStream(new ByteArrayInputStream(data)) {\n            @Override\n            protected Class<?> resolveClass(ObjectStreamClass desc) throws IOException, ClassNotFoundException {\n                if (!desc.getName().equals(\"com.example.SafeClass\")) {\n                    throw new InvalidClassException(\"Unauthorized deserialization attempt\", desc.getName());\n                }\n                return super.resolveClass(desc);\n            }\n        };\n        return ois.readObject();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class WeakHashFunction {\n    public String hashPassword(String password) throws NoSuchAlgorithmException {\n        MessageDigest md = MessageDigest.getInstance(\"MD5\");\n        byte[] hash = md.digest(password.getBytes());\n        return Base64.getEncoder().encodeToString(hash);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class StrongHashFunction {\n    public String hashPassword(String password) throws NoSuchAlgorithmException {\n        MessageDigest md = MessageDigest.getInstance(\"SHA-256\");\n        SecureRandom random = new SecureRandom();\n        byte[] salt = new byte[16];\n        random.nextBytes(salt);\n        md.update(salt);\n        byte[] hash = md.digest(password.getBytes());\n        return Base64.getEncoder().encodeToString(hash);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class LogInjection {\n    public void logUserInput(String userInput) {\n        logger.info(\"User input: \" + userInput);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class SecureLogging {\n    public void logUserInput(String userInput) {\n        String sanitized = userInput.replaceAll(\"[\\\\r\\\\n]\", \"_\");\n        logger.info(\"User input: {}\", sanitized);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class FileUploadVulnerable {\n    public void uploadFile(String fileName, byte[] content) throws IOException {\n        File file = new File(\"/uploads/\" + fileName);\n        FileOutputStream fos = new FileOutputStream(file);\n        fos.write(content);\n        fos.close();\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class FileUploadSecure {\n    public void uploadFile(String fileName, byte[] content) throws IOException {\n        if (!fileName.matches(\"^[a-zA-Z0-9._-]+\\\\.(jpg|jpeg|png|gif|pdf|txt)$\")) {\n            throw new IllegalArgumentException(\"Invalid file type\");\n        }\n        if (content.length > 1024 * 1024) { // 1MB limit\n            throw new IllegalArgumentException(\"File too large\");\n        }\n        File file = new File(\"/uploads/\" + UUID.randomUUID() + \"_\" + fileName);\n        FileOutputStream fos = new FileOutputStream(file);\n        fos.write(content);\n        fos.close();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class ReflectionVulnerable {\n    public void executeMethod(String className, String methodName) throws Exception {\n        Class<?> clazz = Class.forName(className);\n        Method method = clazz.getMethod(methodName);\n        method.invoke(clazz.newInstance());\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class ReflectionSecure {\n    private static final Set<String> ALLOWED_CLASSES = Set.of(\"com.example.SafeClass\");\n    \n    public void executeMethod(String className, String methodName) throws Exception {\n        if (!ALLOWED_CLASSES.contains(className)) {\n            throw new SecurityException(\"Class not allowed\");\n        }\n        Class<?> clazz = Class.forName(className);\n        Method method = clazz.getMethod(methodName);\n        method.invoke(clazz.newInstance());\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class BasicMathOperation {\n    public int add(int a, int b) {\n        return a + b;\n    }\n    \n    public int multiply(int a, int b) {\n        return a * b;\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class StringUtility {\n    public String reverse(String input) {\n        if (input == null) {\n            return null;\n        }\n        return new StringBuilder(input).reverse().toString();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    }
]
qwen_testcases=[
    {
        "code": "public class Example1 {\n    public static void main(String[] args) {\n        String secretKey = \"ABCD1234\";\n        System.out.println(\"Secret: \" + secretKey);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example2 {\n    public static void main(String[] args) {\n        try {\n            Runtime.getRuntime().exec(\"rm -rf /tmp/*\");\n        } catch (Exception e) {\n            e.printStackTrace();\n        }\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.*;\npublic class Example3 {\n    public static void main(String[] args) throws IOException {\n        FileInputStream fis = new FileInputStream(\"/tmp/data.txt\");\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example4 {\n    public static void main(String[] args) {\n        String userInput = args[0];\n        System.out.println(\"Input: \" + userInput);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.*;\npublic class Example5 {\n    public static void main(String[] args) throws Exception {\n        ObjectInputStream ois = new ObjectInputStream(new FileInputStream(args[0]));\n        Object obj = ois.readObject();\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example6 {\n    public static void main(String[] args) {\n        final String apiKey = System.getenv(\"API_KEY\");\n        System.out.println(\"Using API key...\");\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example7 {\n    public static void main(String[] args) {\n        if (args.length > 0 && args[0].equals(\"admin\")) {\n            System.out.println(\"Access granted\");\n        }\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example8 {\n    public static void main(String[] args) {\n        String password = \"password123\";\n        System.out.println(\"Password: \" + password);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.*;\npublic class Example9 {\n    public static void main(String[] args) throws Exception {\n        String filename = args[0];\n        if (!filename.endsWith(\".txt\")) {\n            throw new IllegalArgumentException(\"Only .txt files allowed\");\n        }\n        BufferedReader br = new BufferedReader(new FileReader(filename));\n        String line;\n        while ((line = br.readLine()) != null) {\n            System.out.println(line);\n        }\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example10 {\n    public static void main(String[] args) {\n        ProcessBuilder pb = new ProcessBuilder(args);\n        try {\n            pb.start();\n        } catch (Exception e) {\n            e.printStackTrace();\n        }\n    }\n}",
        "expected": "VULNERABLE"
    }
]
grok_testcases=[
    {
        "code": "public class Example1 {\n    public static void main(String[] args) {\n        String userInput = args[0];\n        Runtime.getRuntime().exec(userInput);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example2 {\n    public static void main(String[] args) {\n        String safeString = \"Hello, World!\";\n        System.out.println(safeString);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example3 {\n    public static void main(String[] args) {\n        String query = \"SELECT * FROM users WHERE id = '\" + args[0] + \"'\";\n        Statement stmt = connection.createStatement();\n        stmt.executeQuery(query);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example4 {\n    public static void main(String[] args) {\n        String input = args[0];\n        if (input.matches(\"[a-zA-Z0-9]+\")) {\n            System.out.println(input);\n        } else {\n            throw new IllegalArgumentException(\"Invalid input\");\n        }\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example5 {\n    public static void main(String[] args) throws Exception {\n        String path = \"../../etc/passwd\";\n        File file = new File(path);\n        FileInputStream fis = new FileInputStream(file);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example6 {\n    public static void main(String[] args) {\n        int number = Integer.parseInt(args[0]);\n        System.out.println(\"Number: \" + number);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example7 {\n    public static void main(String[] args) {\n        String[] allowedValues = {\"option1\", \"option2\", \"option3\"};\n        String input = args[0];\n        for (String value : allowedValues) {\n            if (value.equals(input)) {\n                System.out.println(\"Valid option: \" + input);\n                return;\n            }\n        }\n        System.out.println(\"Invalid option\");\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example8 {\n    public static void main(String[] args) {\n        String secret = \"hardcoded-key-12345\";\n        System.out.println(secret);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example9 {\n    public static void main(String[] args) throws Exception {\n        String input = args[0];\n        PreparedStatement pstmt = connection.prepareStatement(\"SELECT * FROM users WHERE id = ?\");\n        pstmt.setString(1, input);\n        pstmt.executeQuery();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "public class Example10 {\n    public static void main(String[] args) {\n        String xml = args[0];\n        DocumentBuilderFactory factory = DocumentBuilderFactory.newInstance();\n        DocumentBuilder builder = factory.newDocumentBuilder();\n        Document doc = builder.parse(new InputSource(new StringReader(xml)));\n    }\n}",
        "expected": "VULNERABLE"
    }
]
chatgpt_testcases=[
    {
        "code": "public class Example1 {\n    public static void main(String[] args) {\n        String password = \"admin123\";\n        System.out.println(\"Password is: \" + password);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.sql.*;\npublic class Example2 {\n    public void getUser(String username) throws Exception {\n        Connection conn = DriverManager.getConnection(\"jdbc:mysql://localhost/test\", \"root\", \"\");\n        Statement stmt = conn.createStatement();\n        String query = \"SELECT * FROM users WHERE username = '\" + username + \"'\";\n        ResultSet rs = stmt.executeQuery(query);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.io.*;\npublic class Example3 {\n    public static void main(String[] args) throws IOException {\n        BufferedReader br = new BufferedReader(new InputStreamReader(System.in));\n        String line = br.readLine();\n        System.out.println(\"You entered: \" + line);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import javax.crypto.Cipher;\nimport javax.crypto.spec.SecretKeySpec;\npublic class Example4 {\n    public void encrypt(String data) throws Exception {\n        byte[] key = \"1234567890123456\".getBytes();\n        SecretKeySpec secretKey = new SecretKeySpec(key, \"AES\");\n        Cipher cipher = Cipher.getInstance(\"AES\");\n        cipher.init(Cipher.ENCRYPT_MODE, secretKey);\n        cipher.doFinal(data.getBytes());\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example5 {\n    public static void main(String[] args) {\n        String fileName = args[0];\n        if (!fileName.matches(\"[a-zA-Z0-9]+\\\\\\.txt\")) {\n            throw new IllegalArgumentException(\"Invalid file name\");\n        }\n        System.out.println(\"File name is valid: \" + fileName);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.io.*;\npublic class Example6 {\n    public void readFile(String filePath) throws IOException {\n        File file = new File(filePath);\n        FileReader fr = new FileReader(file);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "public class Example7 {\n    public static void main(String[] args) {\n        int a = 5;\n        int b = 10;\n        int sum = a + b;\n        System.out.println(\"Sum: \" + sum);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.net.*;\npublic class Example8 {\n    public void connect(String host) throws Exception {\n        Socket socket = new Socket(host, 80);\n    }\n}",
        "expected": "VULNERABLE"
    },
    {
        "code": "import java.util.*;\npublic class Example9 {\n    public static void main(String[] args) {\n        List<String> list = new ArrayList<>();\n        list.add(\"A\");\n        list.add(\"B\");\n        System.out.println(list);\n    }\n}",
        "expected": "NOT_VULNERABLE"
    },
    {
        "code": "import java.sql.*;\npublic class Example10 {\n    public void safeQuery(String username) throws Exception {\n        Connection conn = DriverManager.getConnection(\"jdbc:mysql://localhost/test\", \"root\", \"\");\n        String query = \"SELECT * FROM users WHERE username = ?\";\n        PreparedStatement ps = conn.prepareStatement(query);\n        ps.setString(1, username);\n        ResultSet rs = ps.executeQuery();\n    }\n}",
        "expected": "NOT_VULNERABLE"
    }
]
mistral_testcases=[{
        "code": 'public class Example1 {\n    public static void main(String[] args) {\n        String password = "myPassword123";\n        System.out.println("Password is: " + password);\n    }\n}',
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example2 {\n    public static void main(String[] args) throws Exception {\n        String userInput = args[0];\n        String query = "SELECT * FROM users WHERE username = \\\'" + userInput + "\\\'";\n        // Execute query\n    }\n}',
        "expected": "VULNERABLE",
    },
    {
        "code": 'import java.io.File;\n\npublic class Example3 {\n    public static void main(String[] args) throws Exception {\n        String fileName = args[0];\n        File file = new File("data/" + fileName);\n        // Read file\n    }\n}',
        "expected": "VULNERABLE",
    },
    {
        "code": 'public class Example4 {\n    public static void main(String[] args) {\n        String userInput = args[0];\n        System.out.println(userInput);\n    }\n}',
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'import java.sql.*;\n\npublic class Example5 {\n    public static void main(String[] args) throws Exception {\n        String userInput = args[0];\n        Connection conn = DriverManager.getConnection("jdbc:mysql://localhost/db", "user", "pass");\n        PreparedStatement stmt = conn.prepareStatement("SELECT * FROM users WHERE username = ?");\n        stmt.setString(1, userInput);\n        ResultSet rs = stmt.executeQuery();\n    }\n}',
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'import java.io.File;\n\npublic class Example6 {\n    public static void main(String[] args) throws Exception {\n        String fileName = args[0];\n        if (!fileName.matches("[a-zA-Z0-9]+\\\\\\\\.txt")) {\n            throw new IllegalArgumentException("Invalid file name");\n        }\n        File file = new File("data/" + fileName);\n        // Read file\n    }\n}',
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example7 {\n    public static void main(String[] args) {\n        String userInput = args[0];\n        if (!userInput.matches("[a-zA-Z0-9]+")) {\n            throw new IllegalArgumentException("Invalid input");\n        }\n        System.out.println(userInput);\n    }\n}',
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example8 {\n    public static void main(String[] args) {\n        String apiKey = "SECRET_API_KEY_123";\n        System.out.println(apiKey);\n    }\n}',
        "expected": "VULNERABLE",
    },
    {
        "code": 'import java.io.*;\n\npublic class Example9 {\n    public static void main(String[] args) throws IOException {\n        BufferedReader reader = new BufferedReader(new InputStreamReader(System.in));\n        String input = reader.readLine();\n        System.out.println("You entered: " + input);\n    }\n}',
        "expected": "NOT_VULNERABLE",
    },
    {
        "code": 'public class Example10 {\n    public static void main(String[] args) throws Exception {\n        Runtime.getRuntime().exec(args[0]);\n    }\n}',
        "expected": "VULNERABLE",
    }]

all_testcases = []

for item in grok_testcases:
  all_testcases.append(item)
for item in qwen_testcases:
  all_testcases.append(item)
for item in gemini_testcases:
  all_testcases.append(item)
for item in claude_testcases:
  all_testcases.append(item)
for item in chatgpt_testcases:
  all_testcases.append(item)
for item in mistral_testcases:
  all_testcases.append(item)


In [ ]:
result=evaluate_model("deepseek",deepseek_test)

In [ ]:
result=evaluate_model("gemini",gemini_testcases)

In [ ]:
result=evaluate_model("claude",claude_testcases)

In [ ]:
result=evaluate_model("qwen", qwen_testcases)

In [ ]:
result=evaluate_model("chatgpt",chatgpt_testcases)

In [ ]:
result=evaluate_model("grok",grok_testcases)

In [ ]:
result=evaluate_model("mistral",mistral_testcases)

In [ ]:
len(all_testcases)

In [ ]:
results=evaluate_model("all testcases", all_testcases)

In [ ]:
results=evaluate_model("all testcases", all_testcases)

In [ ]:
results=evaluate_model("all testcases", all_testcases)

In [ ]:
# @title
import json
import javalang
from typing import Dict, List
import re
TOKENIZER=tokenizer
# Assume TOKENIZER is already defined by the user
# TOKENIZER should support .encode() and have a model_max_length attribute

def clean_code(code: str) -> str:
    """Clean Java code snippet by removing markdown, normalizing whitespace, and standardizing indentation."""
    code = re.sub(r'```java\n|```', '', code).strip()
    lines = code.split('\n')
    cleaned_lines = [line.rstrip() for line in lines if line.strip()]
    indented_lines = ['    ' + line.lstrip() if line.strip() else line for line in cleaned_lines]
    return '\n'.join(indented_lines)

def parse_java_code(code: str) -> Dict:
    """Parse Java code using javalang to extract class/method signatures, control structures, and sensitive APIs."""
    try:
        tree = javalang.parse.parse(code)
        metadata = {
            "classes": [],
            "methods": [],
            "control_structures": [],
            "sensitive_api_calls": [],
            "variables": []
        }
        for path, node in tree:
            if isinstance(node, javalang.tree.ClassDeclaration):
                metadata["classes"].append(node.name)
            elif isinstance(node, javalang.tree.MethodDeclaration):
                metadata["methods"].append({
                    "name": node.name,
                    "parameters": [param.name for param in node.parameters],
                    "return_type": node.return_type.name if node.return_type else "void"
                })
            elif isinstance(node, javalang.tree.IfStatement):
                metadata["control_structures"].append("if")
            elif isinstance(node, javalang.tree.ForStatement):
                metadata["control_structures"].append("for")
            elif isinstance(node, javalang.tree.WhileStatement):
                metadata["control_structures"].append("while")
            elif isinstance(node, javalang.tree.MethodInvocation):
                sensitive_apis = ['System.out.println', 'DriverManager.getConnection', 'ThreadLocal.get', 'Runtime.exec']
                if node.member in sensitive_apis:
                    metadata["sensitive_api_calls"].append(node.member)
            elif isinstance(node, javalang.tree.VariableDeclaration):
                var_type = node.type.name if node.type else "unknown"
                for declarator in node.declarators:
                    metadata["variables"].append({
                        "name": declarator.name,
                        "type": var_type
                    })
        return metadata
    except (javalang.parser.JavaSyntaxError, javalang.tokenizer.LexerError) as e:
        print(f"Parsing error ({type(e).__name__}) during parsing: {e}. Code snippet: \n{code[:200]}...")
        raise
    except TypeError as e:
        print(f"TypeError during parsing: {e}. Code snippet: \n{code[:200]}...")
        raise

def segment_code(code: str, max_tokens: int = 128000) -> List[str]:
    """Segment code into method-level units, ensuring each fits within token limits."""
    try:
        tree = javalang.parse.parse(code)
        methods = []
        for path, node in tree.filter(javalang.tree.MethodDeclaration):
            method_code = f"{node.return_type.name if node.return_type else 'void'} {node.name}() {{ {str(node.body) if node.body else ''} }}"
            token_count = len(TOKENIZER.encode(method_code))
            if token_count <= max_tokens:
                methods.append(method_code)
        if not methods:
            token_count = len(TOKENIZER.encode(code))
            if token_count <= max_tokens:
                methods = [code]
            else:
                methods = [code[:TOKENIZER.model_max_length]]  # Truncate to max length
        return methods
    except (javalang.parser.JavaSyntaxError, javalang.tokenizer.LexerError) as e:
        print(f"Parsing error ({type(e).__name__}) during segmentation: {e}. Code snippet: \n{code[:200]}...")
        token_count = len(TOKENIZER.encode(code))
        if token_count <= max_tokens:
            return [code]
        return [code[:TOKENIZER.model_max_length]]
    except TypeError as e:
        print(f"TypeError during segmentation: {e}. Code snippet: \n{code[:200]}...")
        raise

def simple_data_flow_analysis(code: str) -> Dict:
    """Perform basic data flow analysis to identify taint sources and sinks."""
    taint_sources = []
    taint_sinks = []
    source_patterns = [r'\bSystem\.in\b', r'\bgetParameter\b']
    sink_patterns = [r'\bexecuteQuery\b', r'\bSystem\.out\.println\b', r'\bgetConnection\b', r'\bRuntime\.exec\b']
    for pattern in source_patterns:
        if re.search(pattern, code):
            taint_sources.append(pattern)
    for pattern in sink_patterns:
        if re.search(pattern, code):
            taint_sinks.append(pattern)
    return {"taint_sources": taint_sources, "taint_sinks": taint_sinks}

def process_jsonl_entry(entry: Dict) -> Dict:
    """Process a single entry, using original code on error."""
    code = entry["code"]
    expected=entry["expected"]
    try:
        cleaned_code = clean_code(code)
        parsed_metadata = parse_java_code(cleaned_code)
        segmented_code = segment_code(cleaned_code)
        data_flow = simple_data_flow_analysis(cleaned_code)

        # Store processed content in messages
        messages = json.dumps({
            "code": cleaned_code,
            "metadata": {
                **parsed_metadata,
                "data_flow": data_flow
            }
        })
        code_content = messages
    except Exception as e:
        print(f"Error processing entry: {e}. Using original code.")
        code_content = code  # Use original code on any error

    # Structure prompt_content
    prompt_content = {
        "code": code_content,
        "expected":expected
    }
    return prompt_content

def main(deepseek_test: List[Dict], output_file: str = None) -> List[Dict]:
    """Process deepseek_test list and optionally save to a JSONL file."""
    processed_entries = []
    for entry in deepseek_test:
        try:
            processed_entry = process_jsonl_entry(entry)
            if processed_entry:
                processed_entries.append(processed_entry)
        except Exception as e:
            print(f"Error processing entry: {e}. Skipping entry.")

    if output_file:
        with open(output_file, 'w', encoding='utf-8') as f:
            for entry in processed_entries:
                f.write(json.dumps(entry) + '\n')

    return processed_entries

if __name__ == "__main__":

    processed = main(all_testcases, "processed_all_test.jsonl")

    for entry in processed:

        print(json.dumps(entry, indent=2))

In [ ]:
results=evaluate_model("processed_new", processed)

In [ ]:
from google.colab import userdata
hf_token=userdata.get('WRITE_HF_TOKEN')
model.push_to_hub("barryallen16/Qwen2.5_7B_java_vul_v3", token = hf_token) 
tokenizer.push_to_hub("barryallen16/Qwen2.5_7B_java_vul_v3", token = hf_token) 